# Workflow YAML Validation

This notebook validates BioSeqDownloader workflow YAML descriptors without making API calls. It is safe to run offline as long as BioSeqDownloader and its local dependencies are importable.

The examples use repository-relative paths and the frozen `workflow-v1` schema.

In [5]:
from pathlib import Path
from pprint import pprint

from bioseq_dl.cli.workflows import load_workflow_recipe, validate_workflow_recipe

repo_root = Path.cwd().parent.parent
workflow_dir = repo_root / "examples" / "workflows"
invalid_dir = workflow_dir / "invalid"

print(workflow_dir)

c:\Users\Rosem\Documents\PythonProjects\BioSeqDownloader\examples\workflows


## Validate workflow-v1 examples

The valid descriptors under `examples/workflows/` should include `schema_version: "workflow-v1"` and validate without executing any APIs.

In [6]:
valid_paths = sorted(path for path in workflow_dir.glob("*.yml"))
validated = {}

for config_path in valid_paths:
    recipe = load_workflow_recipe(config_path)
    validated[config_path.name] = validate_workflow_recipe(recipe)

print(f"Validated {len(validated)} workflow descriptors:")
pprint(list(validated))

Validated 8 workflow descriptors:
['compound_chembl_ic50_ranges.yml',
 'full_options_reference.yml',
 'interaction_pli_chembl.yml',
 'interaction_ppi_uniprot.yml',
 'protein_query_composition_labels.yml',
 'protein_query_first_minimal.yml',
 'protein_query_first_with_fields.yml',
 'protein_query_first_with_gui_metadata.yml']


## Inspect normalized values

Validation normalizes executable values while preserving descriptive metadata.

In [7]:
example_name = "protein_query_first_minimal.yml"
normalized = validated[example_name]

pprint({
    "schema_version": normalized["schema_version"],
    "query": normalized["query"],
    "modality": normalized["modality"],
    "mode": normalized["mode"],
    "export_format": normalized["export_format"],
    "output": normalized["output"],
})

{'export_format': 'csv',
 'modality': 'protein',
 'mode': 'query_first',
 'output': 'examples/results/protein_query_first_minimal',
 'query': 'gene:TP53 AND reviewed:true AND organism:human',
 'schema_version': 'workflow-v1'}


## Validate intentionally invalid examples

The files under `examples/workflows/invalid/` are expected to fail validation. These examples demonstrate common schema errors without making API calls.

In [8]:
invalid_results = {}

for config_path in sorted(invalid_dir.glob("*.yml")):
    try:
        recipe = load_workflow_recipe(config_path)
        validate_workflow_recipe(recipe)
    except (TypeError, ValueError) as exc:
        invalid_results[config_path.name] = str(exc)
    else:
        invalid_results[config_path.name] = "Unexpectedly valid"

pprint(invalid_results)

{'forbidden_version_key.yml': "Unknown workflow YAML key 'version'. Use "
                              'schema_version: "workflow-v1".',
 'invalid_query_composition.yml': 'Workflow YAML key '
                                  "'query.composition[0].label' must be a "
                                  'non-empty string.',
 'missing_schema_version.yml': 'Workflow YAML is missing required top-level '
                               "key 'schema_version'. Use schema_version: "
                               '"workflow-v1".',
 'unknown_top_level_section.yml': "Unknown workflow YAML section 'resoures'. "
                                  'Allowed sections are: schema_version, '
                                  'dataset, query, resources, execution, '
                                  'harmonization, export, reporting, '
                                  'interaction_retrieval, activity_retrieval, '
                                  'chemical_metadata_integration, '
                         